# Meqpy Tutorial — 4. BandSystem and BandTransitions

[← Previous: 3. Simulating Experiments](03_Simulating_Experiments.ipynb) | [🏠 Index](00_Overview.ipynb) | [Next: 5a. Cubes and Transitions →](05a_Cubes_and_Transitions.ipynb)

In [ ]:
import meqpy

import numpy as np
import matplotlib.pyplot as plt

### Overview

- [4. BandSystem and BandTransitions](#bands)
    - [4.1 BandTransition](#bands_transition)
    - [4.2 BandSystem](#bands_system)
        - [4.2.1. Add BandTransitions](#bands_add_bands)
        - [4.2.2 Charging Rates in Band Transitions](#bands_charging_rates)
        - [4.2.3 Kappa Mode and Extrapolation](#bands_extrapolation)
    - [4.3 Example: I(V)](#bands_experiment)
        - [4.3.1 System with Transport Gap](#bands_experiment_gapped)
        - [4.3.2 Metallic System](#bands_experiment_metallic)

<a id='bands'></a>
## 4. BandSystem and BandTransition

The standard ``System`` class is limited to resonant tunneling only, meaning the tunneling rate does not increase with higher absolute bias voltages (neglecting the effect of finite bias voltages on the decay constant). For bulk or 2D systems, the density of states (DoS) is not limited to single discrete energies, but instead features electronic band structures with dispersion relations. The number of states one would need to consider in such a system becomes infinite, which poses computational challenges.

The ``BandTransition`` class, in combination with ``BandSystem``, allows one to mimic band-like states with a dispersion relation, under two assumptions:
- only tunneling near the band maximum/minimum is considered, since the bands are approximated by a parabola in $k$-space
- once a charge carrier is injected anywhere into a band, it relaxes instantly to the band maximum/minimum

<a id='bands_transition'></a>
### 4.1 BandTransition

The ``BandTransition`` class is a data class, similar to the ``State`` class and handles most of the properties needed to describe an electronic band maximum/minimum. Its input parameter are:
- ``kpar_offset``: $k_∥$ of the band maximum/minimum in Å<sup>-1</sup> , default is 0.
- ``effective_mass``: effective mass to describe the band curvature, in units of electron mass, can be negative, default is 1.
- ``bandwidth``: energy range of band, starting at maximum/minimum in units of eV, default is 1.
- ``e_offset``: shift the vertex of the parabola down in energy, default is 0.
- ``hwhm``: if ``hwhm > 0``: broaden the band charging rate with gaussian and ``hwhm`` in eV, default is 0.
- ``dx``: energy grid density to calculate the band charging rate in eV, default is 1e-3

In [ ]:
band = meqpy.BandTransition(
    kpar_offset=1.5,  # 1/Å
    effective_mass=1.5,  # m_e
    bandwidth=1.5,  # eV
    e_offset=0.0,  # eV
    hwhm=50e-3,  # eV
)

A ``BandTransition`` object has three important properties:
- ``energy``: energy grid for ``kpar`` and ``dos``
- ``kpar``: $k_∥$ dispersion relation of the band
- ``dos``: Density of states of the band: ``self.effective_mass`` inside the band, 0 otherwise

Note 1: The energy grid used within the ``BandTransition`` class assumes always a parabolic band with minimum at ``e_offset`` and opening to positive energy values. The shifting and inverting of the band parabola, in accordance with charge and energy difference, will be handled within the ``BandSystem`` class.

Note 2: In case ``hwhm > 0``, the energy range will be increased by ``10 hwhm`` to allow for gaussian broadening later. The bandwidth of the band is unaffected.

In [ ]:
energy = band.energy
kpar = band.kpar
dos = band.dos

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))

ax.plot(energy, kpar, color="k")
ax.set_ylim(-0.05, 1.6)
ax.set_xlabel("Energy (eV)")
ax.set_ylabel(r"$k_{par}$ (1/Å)")

ax2 = ax.twinx()
ax2.plot(energy, dos, color="r")
ax2.set_xlabel("Energy (eV)")
ax2.set_ylabel("Density of states")
ax2.yaxis.label.set_color("red")
ax2.tick_params(axis="y", colors="red")

plt.tight_layout(pad=1)
plt.show()

<a id='bands_system'></a>
### 4.2 BandSystem

The ``BandSystem`` class is an extension of the ``System`` class, with some added and modified methods to handle ``BandTransitions``.
There are two major differences in how charging transitions are handled, compared to ``System``:
1. The decay constant κ (``BandSystem.kappa()``) is expanded to include the parallel wavevector:
    ```math
        \kappa(k_∥) = \sqrt{\kappa_0^2 + k_∥^2}
    ```
1. Charging transitions from state *a* to state *b*, for which a ``BandTransition`` object is registered, are calculated based on that object.
    - The transition from *b* to *a* is treated with the default peak-like DoS, but includes k<sub>∥</sub> at the band maximum/minimum.

The ``BandSystem`` class is initialized in the same way, with the same arguments, as the ``System`` class:

In [ ]:
bandsystem = meqpy.BandSystem(
    hwhm=0.05,  # eV
    kappa_mode="constant",
)

bandsystem.states = [
    meqpy.State(label="GS", energy=0.0, charge=0, multiplicity=1),
    meqpy.State(label="CB", energy=0.5, charge=-1, multiplicity=2),
    meqpy.State(label="VB", energy=1.0, charge=+1, multiplicity=2),
]

<a id='bands_add_bands'></a>
#### 4.2.1 Add BandTransitions to BandSystem

``BandTransition`` objects need to be added manually to the corresponding state transitions in the band system. This requires careful consideration by the user.

In the example above, we created a system with three states:
- "GS": the ground state of the system, with fully occupied valence bands and empty conduction bands
- "CB": the same as "GS" but with one additional electron at the band minimum of the conduction band
- "VB": the same as "GS" but with one additional hole at the band maximum of the valence band

Accordingly, we need to consider two combinations of states, each with two transitions (forward and backward):
- "GS" ↔ "CB":
    1. "GS" → "CB": the final state can be reached by injecting an electron anywhere into the conduction band
    1. "CB" → "GS": the electron can only tunnel out from the minimum of the conduction band, since we assume instant relaxation within the bands
- "GS" ↔ "VB":
    1. "GS" → "VB": the final state can be reached by injecting a hole anywhere into the valence band
    1. "VB" → "GS": the hole can only tunnel out from the maximum of the valence band

The "CB" ↔ "VB" transition is forbidden, since it would require more than one electron to tunnel into or out of the system.


``BandTransition`` objects can be added in two ways to a band system:
1. one by one, using ``add_band_transition(<initial>, <final>, <BandTransition>)``
1. as a dictionary, with ``(<initial>, <final>)`` being the keys

In [ ]:
# create bands
CB_transition = meqpy.BandTransition(effective_mass=0.5, kpar_offset=1.2, hwhm=50e-3)
VB_transition = meqpy.BandTransition(effective_mass=1.5, kpar_offset=0.0, hwhm=50e-3)

# Option 1: on-by-one
bandsystem.add_band_transition("GS", "CB", CB_transition)
bandsystem.add_band_transition("GS", "VB", VB_transition)

# Option 2: as dictionary
bandsystem.band_transition_dict = {
    ("GS", "CB"): CB_transition,
    ("GS", "VB"): VB_transition,
}

# the "GS" ↔ "CB" transition has an k_parallel component now
bandsystem.kpar_offsets

In [ ]:
# accordingly, the decay constant increases
# for the "GS" ↔ "CB" transition
bandsystem.kappa(bias=0.0)  # kappa_mode = "constant"

<a id='bands_charging_rates'></a>
#### 4.2.2 Charging Rates in Band Transitions

The charging rates, including band-like transitions, are obtained by the ``BandSystem.charging_rates()`` method, just like for the standard ``System`` class.

In [ ]:
z = 4.0  # Å
bias = np.linspace(-3, 2.5, 301)  # V

charging_rates = bandsystem.charging_rates(z, bias)

charging_rates.shape

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7, 3))

i, j, k = 0, 1, 2
a = bandsystem.get_state(i).label
b = bandsystem.get_state(j).label
c = bandsystem.get_state(k).label

ax[0].plot(bias, charging_rates[:, j, i], label=f"{a} → {b}")
ax[0].plot(bias, charging_rates[:, i, j], label=f"{b} → {a}")
ax[0].set_title(f"{a} ↔ {b}")

ax[1].plot(bias, charging_rates[:, k, i], label=f"{a} → {c}")
ax[1].plot(bias, charging_rates[:, i, k], label=f"{c} → {a}")
ax[1].set_title(f"{a} ↔ {c}")

for i in range(2):
    ax[i].set_xlabel("Bias Voltage (V)")
    ax[i].set_ylabel("Charging Rates (1/s)")
    ax[i].legend(frameon=False)
plt.tight_layout(pad=1)
plt.show()

In the following, there is a rough explanation of the under-the-hood mechanics, even though the user does not need to carry out these steps themselves.

The ``BandSystem.charging_rates()`` method works in three steps:
1. Create the charging rates array ``W``, just like the ``System`` class does, but using $\kappa(k_∥) = \sqrt{\kappa_0^2 + k_∥^2}$
1. For each registered band transition ``(i, f)``, recalculate the charging rate based on the corresponding ``BandTransition`` object
1. Multiply charging rate with corresponding Clebsch-Gordan prefactor and insert it into ``W[f,i]``

For the second step, the ``BandSystem.get_band_charging_rate()`` method is called, which works in several steps:
1. Calculate the decay constant κ as a function of the dispersion relation $k_∥(E)$
    - in case of ``kappa_mode = 'full'``: $\kappa\left( k_∥(E) \right ) \rightarrow \kappa\left ( E, V, k_∥(E) \right )$
1. Calculate the local density of states (LDOS) at the tip height: dos ⋅ e<sup>-2κz</sup>
1. If ``hwhm > 0``: apply Gaussian broadening
1. Integrate over the LDOS and multiply by the conductance quantum G<sub>0</sub> to obtain charging rates in s<sup>-1</sup>
1. Interpolate the charging rates to obtain the rate at the precise bias voltage

In the following, we will repeat these steps in a simplified manner.

In [ ]:
# 1. Calculate decay constant
energy_CB, kappa_CB = bandsystem.band_kappa(("GS", "CB"), bias)
energy_VB, kappa_VB = bandsystem.band_kappa(("GS", "VB"), bias)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))

ax.plot(energy_CB, kappa_CB, label="CB")
ax.plot(energy_VB, kappa_VB, label="VB")
ax.set_xlabel("Energy (eV)")
ax.set_ylabel("Decay Constant 1/Å")
ax.legend(frameon=False)

plt.show()

In [ ]:
# 2. calculate LDOS

ldos_CB = np.exp(-2 * np.multiply.outer(z, kappa_CB)) * CB_transition.dos
ldos_VB = np.exp(-2 * np.multiply.outer(z, kappa_VB)) * VB_transition.dos

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))

ax.plot(energy_CB, ldos_CB * 1e3, label="CB")
ax.plot(energy_VB, ldos_VB * 1e3, label="VB")
ax.set_xlabel("Energy (eV)")
ax.set_ylabel(r"LDOS ($10^{-3}$/eV)")
ax.legend(frameon=False)

plt.show()

In [ ]:
# 3. Gaussian broadened

from scipy.ndimage import gaussian_filter1d


def broadening(ldos, hwhm, dx):
    if hwhm > 0:
        sigma_eV = hwhm / (2 * np.sqrt(2 * np.log(2)))
        sigma_samples = sigma_eV / dx
        return gaussian_filter1d(ldos, sigma_samples, axis=-1)
    return ldos


broad_ldos_CB = broadening(ldos_CB, CB_transition.hwhm, CB_transition.dx)
broad_ldos_VB = broadening(ldos_VB, VB_transition.hwhm, VB_transition.dx)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))

ax.plot(energy_CB, broad_ldos_CB * 1e3, label="CB")
ax.plot(energy_VB, broad_ldos_VB * 1e3, label="VB")
ax.set_xlabel("Energy (eV)")
ax.set_ylabel(r"LDOS ($10^{-3}$/eV)")
ax.legend(frameon=False)

plt.show()

In [ ]:
# 4. integration over energy
G0 = meqpy.constants.G0

rates_CB = np.cumsum(broad_ldos_CB, axis=-1) * CB_transition.dx * G0
rates_VB = np.cumsum(broad_ldos_VB, axis=-1) * VB_transition.dx * G0

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))

ax.plot(energy_CB, rates_CB, label="CB")
ax.plot(energy_VB, rates_VB, label="VB")
ax.set_xlabel("Energy (eV)")
ax.set_ylabel("Rates (1/s)")
ax.legend(frameon=False)

plt.show()

In [ ]:
# 5. Inter- and Extrapolation

from scipy.interpolate import RectBivariateSpline
# RectBivariateSpline is used, because:
# - it allows for multiple interpolation at once
#   which is needed in case of kappa_mode = "full" or multiple z values
# - appears to have the best stable extrapolation


def band_interp(rates, energy, bias):
    if energy[0] > energy[-1]:
        energy = np.flip(energy)
        rates = np.flip(rates, axis=-1)

    rr_flat = np.tile(rates, (2, 1))
    row_indices = np.arange(2)

    interp = RectBivariateSpline(row_indices, energy, rr_flat, kx=1, ky=3)
    return interp(0, bias, grid=True)[0]


band_charging_rates_CB = band_interp(rates_CB, energy_CB, bias)
band_charging_rates_VB = band_interp(rates_VB, energy_VB, bias)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))

ax.plot(bias, band_charging_rates_CB, label="CB")
ax.plot(bias, band_charging_rates_VB, label="VB")
ax.set_xlabel("Energy (eV)")
ax.set_ylabel("Band Charging Rates (1/s)")
ax.legend(frameon=False)

plt.show()

<a id='bands_extrapolation'></a>
#### 4.2.3 Kappa Mode and Extrapolation

For interpolation, the ``RectBivariateSpline`` function from the ``scipy`` package is used, which also allows for reasonably good extrapolation. Nevertheless, there is no guarantee that the extrapolation is reasonable in all cases, especially for ``kappa_mode = "full"``.

In the following, we show the behavior of the extrapolation for the two kappa modes "constant" and "full".

##### Kappa Mode: Constant

In [ ]:
bandsystem.kappa_mode = "constant"

CB_wo_extrap = bandsystem.get_band_charging_rate(
    ("GS", "CB"), z, energy_CB, kappa_mode="constant"
)
CB_w_extrap = bandsystem.get_band_charging_rate(
    ("GS", "CB"), z, bias, kappa_mode="constant"
)

VB_wo_extrap = bandsystem.get_band_charging_rate(
    ("GS", "VB"), z, energy_VB, kappa_mode="constant"
)
VB_w_extrap = bandsystem.get_band_charging_rate(
    ("GS", "VB"), z, bias, kappa_mode="constant"
)

dCB_wo_extrap = np.gradient(CB_wo_extrap, energy_CB) * CB_transition.dx
dCB_w_extrap = np.gradient(CB_w_extrap, bias) * CB_transition.dx

dVB_wo_extrap = np.gradient(VB_wo_extrap, -energy_VB) * VB_transition.dx
dVB_w_extrap = np.gradient(VB_w_extrap, -bias) * VB_transition.dx

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3))

ax[0].plot(bias, CB_w_extrap, label="CB")
ax[0].plot(bias, VB_w_extrap, label="VB")
ax[0].plot(energy_CB, CB_wo_extrap, "--", color="k")
ax[0].plot(energy_VB, VB_wo_extrap, "--", color="k", label="no extrap.")
ax[0].set_ylabel("Band Charging Rates (1/s)")

ax[1].plot(bias, dCB_w_extrap, label="CB")
ax[1].plot(bias, dVB_w_extrap, label="VB")
ax[1].plot(energy_CB, dCB_wo_extrap, "--", color="k")
ax[1].plot(energy_VB, dVB_wo_extrap, "--", color="k", label="no extrap.")
ax[1].set_ylabel("dIdV (e/s)")

for axi in ax:
    axi.set_xlabel("Bias Voltage (eV)")
    axi.legend(frameon=False)
    axi.set_title("Kappa Mode: Constant")

plt.show()

##### Kappa Mode: Full

In [ ]:
CB_wo_extrap = bandsystem.get_band_charging_rate(
    ("GS", "CB"), z, energy_CB, kappa_mode="full"
)
CB_w_extrap = bandsystem.get_band_charging_rate(
    ("GS", "CB"), z, bias, kappa_mode="full"
)

VB_wo_extrap = bandsystem.get_band_charging_rate(
    ("GS", "VB"), z, energy_VB, kappa_mode="full"
)
VB_w_extrap = bandsystem.get_band_charging_rate(
    ("GS", "VB"), z, bias, kappa_mode="full"
)

dCB_wo_extrap = np.gradient(CB_wo_extrap, energy_CB) * CB_transition.dx
dCB_w_extrap = np.gradient(CB_w_extrap, bias) * CB_transition.dx

dVB_wo_extrap = np.gradient(VB_wo_extrap, -energy_VB) * VB_transition.dx
dVB_w_extrap = np.gradient(VB_w_extrap, -bias) * VB_transition.dx

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3))

ax[0].plot(bias, CB_w_extrap, label="CB")
ax[0].plot(bias, VB_w_extrap, label="VB")
ax[0].plot(energy_CB, CB_wo_extrap, "--", color="k")
ax[0].plot(energy_VB, VB_wo_extrap, "--", color="k", label="no extrap.")
ax[0].set_ylabel("Band Charging Rates (1/s)")

ax[1].plot(bias, dCB_w_extrap, label="CB")
ax[1].plot(bias, dVB_w_extrap, label="VB")
ax[1].plot(energy_CB, dCB_wo_extrap, "--", color="k")
ax[1].plot(energy_VB, dVB_wo_extrap, "--", color="k", label="no extrap.")
ax[1].set_ylabel("dIdV (e/s)")

for axi in ax:
    axi.set_xlabel("Bias Voltage (eV)")
    axi.legend(frameon=False)
    axi.set_title("Kappa Mode: Full")

plt.show()

<a id='bands_experiment'></a>
### 4.3 Example: I(V)

In the following we will have a look at two examples: A gapped system and a metallic system (experimental).

<a id='bands_experiment_gapped'></a>
#### 4.3.1 Example: I(V) with Transport Gap + STML

We now consider a system with band-like transitions and an excitonic state. For this, we have to register several ``BandTransition`` objects with our system, but also exclude some charging transitions, since the difference in electron configuration between certain states cannot be compensated by the tunneling of a single electron.

First we initialize our System and add all necessary states:
- "GS": the charge neutral ground state
- "VB_G": removed one electron from the valence band maximum at the Gamma point
- "VB_K": removed one electron from the valence band maximum at the K point
- "CB_K": added an electron to the conduction band minimum at the K point
- "CB_Q": added an electron to the conduction band minimum at the Q point
- "XK": excitonic state at the K point, with one electron in the conduction band and one hole in the valence band

Afterwards, we will add the ``BandTransition`` objects:
- "GS" → *ion resonance*: all transitions from the ground state to a charge state have to be assigned a band in this system
- "VB_K" → "XK": this transition can occur by adding one electron into the conduction band
- "CB_K" → "XK": this transition can occur by injecting one hole into the valence band

Next we need to exclude some transitions, since the transitions "VB_G" ↔ "XK" and "CB_Q" ↔ "XK" are both not allowed. After that, we can run the experiment, as described already above.

In [ ]:
# Initialize System
gapped_system = meqpy.BandSystem(hwhm=0.02, workfunction=4.5, kappa_mode="full")

# ------------------------------------------------------------------------------------
# add states
gapped_system.states = [
    meqpy.State(label="GS", energy=0.00, charge=0, multiplicity=1),
    meqpy.State(label="XK", energy=1.98, charge=0, multiplicity=1),
    meqpy.State(label="VB_G", energy=2.10, charge=+1, multiplicity=2),
    meqpy.State(label="VB_K", energy=1.80, charge=+1, multiplicity=2),
    meqpy.State(label="CB_K", energy=0.35, charge=-1, multiplicity=2),
    meqpy.State(label="CB_Q", energy=0.55, charge=-1, multiplicity=2),
]

# ------------------------------------------------------------------------------------
# Add band transitions
VB_K_transition = meqpy.BandTransition(effective_mass=0.4, kpar_offset=1.32, hwhm=20e-3)
VB_G_transition = meqpy.BandTransition(effective_mass=2.8, kpar_offset=0.00, hwhm=20e-3)
CB_K_transition = meqpy.BandTransition(effective_mass=0.4, kpar_offset=1.32, hwhm=20e-3)
CB_Q_transition = meqpy.BandTransition(effective_mass=0.4, kpar_offset=0.66, hwhm=20e-3)

gapped_system.band_transition_dict = {
    ("GS", "VB_G"): VB_G_transition,
    ("GS", "VB_K"): VB_K_transition,
    ("GS", "CB_K"): CB_K_transition,
    ("GS", "CB_Q"): CB_Q_transition,
    ("VB_K", "XK"): CB_K_transition,
    ("CB_K", "XK"): VB_K_transition,
}

In [ ]:
# exclusion of some transitions:
# set some transition rates to 0, since they are physically not possible
# e.g. "XK" has e and h at K, so no single charging event will transition to "VB_G"
exclusion = gapped_system.ones
exclusion *= gapped_system.rescale_by_states("XK", "VB_G", 0, symmetric=True)
exclusion *= gapped_system.rescale_by_states("XK", "CB_Q", 0, symmetric=True)

exclusion.shape

In [ ]:
# build rate matrix
voltage_drop = 0.1
sample_distance = 4.0
tip_height = 6.0
tau_rad = 10e-12  # ps :  lifetime of excitonic states

bias = np.arange(-3, +1, 0.01)

# ------------------------------------------------------------------------

# tip and sample charging
Wt = gapped_system.charging_rates(
    tip_height, bias * (1 - voltage_drop)
)  # coupling to tip
Ws = gapped_system.charging_rates(
    sample_distance, bias * (0 - voltage_drop)
)  # coupling to sample

# radiative excitation / emission
Wrad = gapped_system.matrix_by_states("XK", "GS") / tau_rad

W = Wt + Ws + Wrad
W *= exclusion

W.shape

In [ ]:
# solve rate equation to obtain probability vector P
gs_P = meqpy.solve_equilibrium(W)

gs_P.shape

In [ ]:
# define measurement operators and perform measurement

# measurement: current
current_operator = gapped_system.dQ * Ws * meqpy.constants.ELEMENTARY_CHARGE
gs_current = meqpy.measurement(current_operator, gs_P)
gs_didv = np.gradient(gs_current, bias, axis=-1)

# measurement: X -> GS emission
x_operator = gapped_system.matrix_by_states("XK", "GS") * 1 / tau_rad
x_emission = meqpy.measurement(x_operator, gs_P)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 3))

# dI/dV axis (left)
ax[0].plot(bias, gs_didv, "-", color="k")
ax[0].set_ylabel("dI/dV (S)")
ax[0].set_yscale("log")
ax[0].set_ylim(5e-14, 5e-9)
ax[0].set_xlabel("Bias (V)")

# Current axis (right)
ax2 = ax[0].twinx()
ax2.plot(bias, gs_current * 1e12, "-", color="r")
ax2.yaxis.label.set_color("red")
ax2.tick_params(axis="y", colors="red")
ax2.set_ylabel("Current (pA)")

ax[1].plot(bias, x_emission, "-")
ax[1].set_ylabel("Emission (1/s)")
ax[1].set_xlabel("Bias (V)")

plt.tight_layout(pad=1)
plt.show()

<a id='bands_experiment_metallic'></a>
#### 4.3.2 Example: I(V) Metallic System

Even though there is probably little benefit in using rate equations to simulate a metallic surface, we will do this example as a finger exercise. Metallic systems are, by definition, not gapped, meaning some bands cross the Fermi energy. To capture this correctly in the Markov chain, we have to split those bands into an occupied and an unoccupied part.

Using the ``e_offset`` attribute of the ``BandTransition`` class, in combination with negative ``effective_mass``, we can create a simplified band structure of a typical noble metal (111) surface. For this we need six states:
- "GS": the ground state
- "VB": the occupied part of the bulk states crossing the Fermi energy
- "CB": the unoccupied part of the bulk states crossing the Fermi energy
- "DB": the high-energy occupied bands corresponding to the $d$-states
- "So": the occupied part of the surface state
- "Su": the unoccupied part of the surface state

The $d$-band "DB" will have an effective mass of 0, to account for the broad band at the $\Gamma$ point. For the bands crossing the Fermi energy we need to consider three things:
1. all states have an energy of 0
1. the occupied part needs a negative ``effective_mass``, since the parabola opens towards the Fermi energy
1. the unoccupied part requires the ``e_offset`` attribute, since the band minimum is part of the occupied band

Since the bands ARE the sample, we will set the ``sample_distance = 0``. In addition, we will manually increase the coupling between the $d$-states ("DB") and the tip to reproduce the experiment.

In [ ]:
# Initialize System
metal = meqpy.BandSystem(
    hwhm=0.0,
    kappa_mode="full",
    workfunction=5.4,
)

# add states
metal.states = [
    meqpy.State("GS", energy=0.0, charge=0, multiplicity=1),
    meqpy.State("VB", energy=0.0, charge=+1, multiplicity=2),
    meqpy.State("CB", energy=0.0, charge=-1, multiplicity=2),
    meqpy.State("DB", energy=1.0, charge=+1, multiplicity=2),
    meqpy.State("So", energy=0.0, charge=+1, multiplicity=2),
    meqpy.State("Su", energy=0.0, charge=-1, multiplicity=2),
]

# add band transitions
metal.band_transition_dict = {
    ("GS", "VB"): meqpy.BandTransition(
        effective_mass=-0.15, bandwidth=1.0, hwhm=0.05, e_offset=0.0
    ),
    ("GS", "CB"): meqpy.BandTransition(
        effective_mass=+0.15, bandwidth=3.0, hwhm=0.05, e_offset=1.0
    ),
    ("GS", "DB"): meqpy.BandTransition(
        effective_mass=+0.00, bandwidth=2.0, hwhm=0.25, e_offset=0.0
    ),
    ("GS", "So"): meqpy.BandTransition(
        effective_mass=-0.20, bandwidth=0.5, hwhm=0.05, e_offset=0.0
    ),
    ("GS", "Su"): meqpy.BandTransition(
        effective_mass=+0.20, bandwidth=3.0, hwhm=0.05, e_offset=0.5
    ),
}

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))

for key, band in metal.band_transition_dict.items():
    energy = metal.band_energy(key)
    mask = band.dos.astype(bool)
    ax.plot(energy[mask], band.kpar[mask], label=key[1])
ax.set_xlabel("Energy (eV)")
ax.set_ylabel(r"$k_∥$ (Å$^{-1}$)")
ax.legend(frameon=False)

plt.show()

In [ ]:
# build rate matrix

sample_distance = 0.0  # the bands ARE the sample
tip_height = 5.0
bias = np.linspace(-2.0, 2.0, 251)

Ws = metal.charging_rates(sample_distance, 0)
Wt = metal.charging_rates(tip_height, bias)

# empirically increase coupling between tip and metallic d-states
Wt *= metal.rescale_by_states("GS", "DB", 4.0, symmetric=True)

W = Ws + Wt

W.shape

In [ ]:
# solve rate equation to obtain probability vector P
ms_P = meqpy.solve_equilibrium(W)

ms_P.shape

In [ ]:
# measurement: current
current_operator = metal.dQ * Ws * meqpy.constants.ELEMENTARY_CHARGE
ms_current = meqpy.measurement(current_operator, ms_P)
ms_didv = np.gradient(ms_current, bias)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))

# dI/dV axis (left)
ax.plot(bias, ms_didv, "-", color="k")
ax.set_ylabel("dI/dV (S)")
ax.set_yscale("log")
ax.set_xlabel("Bias (V)")

# Current axis (right)
ax2 = ax.twinx()
ax2.plot(bias, ms_current * 1e9, "-", color="r")
ax2.yaxis.label.set_color("red")
ax2.tick_params(axis="y", colors="red")
ax2.set_ylabel("Current (nA)")

print("NOTE: The dip at zero bias is an artefact and physically not true.")
plt.show()

---

[← Previous: 3. Simulating Experiments](03_Simulating_Experiments.ipynb) | [🏠 Index](00_Overview.ipynb) | [Next: 5a. Cubes and Transitions →](05a_Cubes_and_Transitions.ipynb)